# Telco Churn - Phase 2: Data Preprocessing and Feature Engineering

This notebook converts raw churn data into model-ready train/test datasets using a leakage-safe pipeline.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.append(str(Path("..").resolve()))

from src.feature_engineering import engineer_all_features
from src.preprocessing import (
    encode_binary_columns,
    encode_multiclass_columns,
    load_and_clean,
    scale_numeric_columns,
)

## Section 1: Load and Clean Raw Data

In [2]:
data_path = Path("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df = load_and_clean(str(data_path))

print("Shape after clean:", df.shape)
print("Columns:", len(df.columns))
print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("Churn distribution:")
print(df["Churn"].value_counts(normalize=True).rename({0: "No", 1: "Yes"}))
df.head()

Shape after clean: (7043, 20)
Columns: 20
TotalCharges dtype: float64
Churn distribution:
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## Section 2: Feature Engineering

In [3]:
df_fe = engineer_all_features(df)

new_cols = [
    "tenure_group",
    "avg_monthly_charge",
    "service_count",
    "charge_per_service",
    "is_new_customer",
    "has_premium_support",
    "contract_risk_score",
]

print("Shape after feature engineering:", df_fe.shape)
print("New columns:", new_cols)
df_fe[new_cols].head()

Shape after feature engineering: (7043, 27)
New columns: ['tenure_group', 'avg_monthly_charge', 'service_count', 'charge_per_service', 'is_new_customer', 'has_premium_support', 'contract_risk_score']


,tenure_group,avg_monthly_charge,service_count,charge_per_service,is_new_customer,has_premium_support,contract_risk_score
0,0-12,29.850000,1,29.850,1,0,4
1,25-48,55.573529,2,28.475,0,0,2
2,0-12,54.075000,2,26.925,1,0,3
3,25-48,40.905556,3,14.100,0,1,1
4,0-12,75.825000,0,70.700,1,0,4


## Section 3: Train/Test Split (Leakage Guardrail)

In [4]:
target_col = "Churn"
X = df_fe.drop(columns=[target_col])
y = df_fe[target_col]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("X_train_raw:", X_train_raw.shape)
print("X_test_raw:", X_test_raw.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True).rename({0: "No", 1: "Yes"}))
print("y_test distribution:")
print(y_test.value_counts(normalize=True).rename({0: "No", 1: "Yes"}))

X_train_raw: (5634, 26)
X_test_raw: (1409, 26)
y_train distribution:
Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64
y_test distribution:
Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


## Section 4: Binary Encoding (Fit on Train, Apply to Test)

In [5]:
binary_columns = [
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
]

X_train_bin, binary_mapping = encode_binary_columns(X_train_raw, binary_columns)
X_test_bin, _ = encode_binary_columns(X_test_raw, binary_columns, mapping=binary_mapping)

print("Binary mapping:")
print(binary_mapping)
X_train_bin[binary_columns].head()

Binary mapping:
{'gender': {'Female': 0, 'Male': 1}, 'SeniorCitizen': {0: 0, 1: 1}, 'Partner': {'No': 0, 'Yes': 1}, 'Dependents': {'No': 0, 'Yes': 1}, 'PhoneService': {'No': 0, 'Yes': 1}, 'PaperlessBilling': {'No': 0, 'Yes': 1}}


,gender,SeniorCitizen,Partner,Dependents,PhoneService,PaperlessBilling
3738,1,0,0,0,0,0
3151,1,0,1,1,1,0
4860,1,0,1,1,0,0
3867,0,0,1,0,1,1
3810,1,0,1,1,1,0


## Section 5: Multi-class One-Hot Encoding (Fit on Train, Transform Test)

In [6]:
multiclass_columns = [
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod",
    "tenure_group",
]

X_train_ohe, X_test_ohe, multiclass_encoder = encode_multiclass_columns(
    X_train_bin,
    X_test_bin,
    multiclass_columns,
)

print("X_train_ohe shape:", X_train_ohe.shape)
print("X_test_ohe shape:", X_test_ohe.shape)
print("OneHot encoded feature count:", len(multiclass_encoder.get_feature_names_out()))

X_train_ohe shape: (5634, 40)
X_test_ohe shape: (1409, 40)
OneHot encoded feature count: 25


## Section 6: Numeric Scaling (Fit on Train, Transform Test)

In [7]:
numeric_columns_to_scale = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "avg_monthly_charge",
    "service_count",
    "charge_per_service",
    "contract_risk_score",
]

X_train_final, X_test_final, scaler = scale_numeric_columns(
    X_train_ohe,
    X_test_ohe,
    numeric_columns_to_scale,
)

print("X_train_final shape:", X_train_final.shape)
print("X_test_final shape:", X_test_final.shape)
X_train_final.head()

X_train_final shape: (5634, 40)
X_test_final shape: (1409, 40)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,avg_monthly_charge,...,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,tenure_group_13-24,tenure_group_25-48,tenure_group_49-60,tenure_group_61+
3738,1,0,0,0,0.130435,0,0,-0.391994,0.089350,-0.403722,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3151,1,0,1,1,-0.304348,1,0,0.084656,-0.070896,0.113588,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
4860,1,0,1,1,-0.347826,0,0,-0.551185,-0.234375,-0.462654,...,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
3867,0,0,1,0,-0.065217,1,1,0.055210,0.148790,0.049753,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3810,1,0,1,1,-0.608696,1,0,-0.477571,-0.393368,-0.478485,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


## Section 7: Verification Checks

In [8]:
print("Train/Test feature columns aligned:", list(X_train_final.columns) == list(X_test_final.columns))
print("Any NaN in train:", X_train_final.isna().any().any())
print("Any NaN in test:", X_test_final.isna().any().any())
print("All train columns numeric:", all(np.issubdtype(dtype, np.number) for dtype in X_train_final.dtypes))
print("All test columns numeric:", all(np.issubdtype(dtype, np.number) for dtype in X_test_final.dtypes))
print("Finite train values:", np.isfinite(X_train_final.to_numpy(dtype=float)).all())
print("Finite test values:", np.isfinite(X_test_final.to_numpy(dtype=float)).all())

print("\nFinal class balance:")
print("y_train:")
print(y_train.value_counts(normalize=True).rename({0: "No", 1: "Yes"}))
print("y_test:")
print(y_test.value_counts(normalize=True).rename({0: "No", 1: "Yes"}))

Train/Test feature columns aligned: True
Any NaN in train: False
Any NaN in test: False
All train columns numeric: True
All test columns numeric: True


Finite train values: True
Finite test values: True

Final class balance:
y_train:
Churn
No     0.734647
Yes    0.265353
Name: proportion, dtype: float64
y_test:
Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


## Section 8: Save Processed Datasets and Artifacts

In [9]:
processed_dir = Path("../data/processed")
models_dir = Path("../models")
processed_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

X_train_path = processed_dir / "X_train.csv"
X_test_path = processed_dir / "X_test.csv"
y_train_path = processed_dir / "y_train.csv"
y_test_path = processed_dir / "y_test.csv"

X_train_final.to_csv(X_train_path, index=False)
X_test_final.to_csv(X_test_path, index=False)
y_train.to_frame(name="Churn").to_csv(y_train_path, index=False)
y_test.to_frame(name="Churn").to_csv(y_test_path, index=False)

joblib.dump(binary_mapping, models_dir / "binary_mapping.joblib")
joblib.dump(multiclass_encoder, models_dir / "multiclass_encoder.joblib")
joblib.dump(scaler, models_dir / "numeric_scaler.joblib")

print("Saved files:")
print("-", X_train_path)
print("-", X_test_path)
print("-", y_train_path)
print("-", y_test_path)
print("-", models_dir / "binary_mapping.joblib")
print("-", models_dir / "multiclass_encoder.joblib")
print("-", models_dir / "numeric_scaler.joblib")

Saved files:
- ../data/processed/X_train.csv
- ../data/processed/X_test.csv
- ../data/processed/y_train.csv
- ../data/processed/y_test.csv
- ../models/binary_mapping.joblib
- ../models/multiclass_encoder.joblib
- ../models/numeric_scaler.joblib
